# Result Plots

In [ ]:
from utils.utils import *
import os
import json
import matplotlib.pyplot as plt
import math

In [ ]:
TOPIC_ORDER = ['politics', 'general', 'covid', 'syria', 'islam', 'notredame', 'gossip'] # Topic order for plots
DATE_ORDER = ['2011-2015', '2016', '2017', '2019', '2020'] # Date order for plots
MODEL_ORDER = ['BERT', 'DeBERTa', 'RoBERTa', 'CNN', 'BiLSTM', 'Linear', 'NB', 'PA'] # Model order for plots

MODEL_COLORS = { # Colors for each model
    'BERT': 'tab:blue',
    'DeBERTa': 'tab:orange',
    'RoBERTa': 'tab:green',
    'CNN': 'tab:red',
    'BiLSTM': 'tab:purple',
    'Linear': 'tab:brown',
    'NB': 'tab:pink',
    'PA': 'tab:gray'
}

MODEL_MARKERS = { # Markers for each model
    'BERT': 'o',
    'DeBERTa': 's',
    'RoBERTa': '^',
    'CNN': 'D',
    'BiLSTM': 'v',
    'Linear': 'P',
    'NB': 'X',
    'PA': '*'
}

plt.rcParams.update({'font.size': 14})

#### Plot Functions

In [ ]:
def load_data(type='topic', cl='all_data'):
    """
    Load full and per-task results for each model.

    Args:
        type (str): 'topic' or 'date'
        cl (str): suffix of the results folder (e.g., 'all_data')

    Returns:
        dict:
        {
            model_name: {
                'full': float,
                'single': {task: value}
            }
        }
    """
    RESULTS_DIR = f'results_{cl}'
    model_data = {}

    for folder_name in os.listdir(RESULTS_DIR):
        folder_path = os.path.join(RESULTS_DIR, folder_name)

        if os.path.isdir(folder_path) and folder_name.endswith(f'_{type}'):
            parts = folder_name.split('_')
            model_name = parts[1]

            full_path = os.path.join(folder_path, f'results_full_{type}.json')
            single_path = os.path.join(folder_path, f'results_single_{type}.json')

            data = {}

            if os.path.exists(full_path):
                with open(full_path, 'r') as f:
                    data['full'] = json.load(f)

            if os.path.exists(single_path):
                with open(single_path, 'r') as f:
                    data['single'] = json.load(f)

            if data:
                model_data[model_name] = data

    ordered_model_data = {m: model_data[m] for m in MODEL_ORDER if m in model_data}
    return ordered_model_data


def plot_full_results(model_data, type='topic'):
    """
    Plot comparison of full test results across models.
    Compatible with both:
    - full = float
    - full = {"f1_score": float}
    """
    models = []
    values = []
    colors = []

    for model_name, data in model_data.items():
        if 'full' not in data:
            continue

        full_data = data['full']

        # NEW: handle dict or float
        if isinstance(full_data, dict):
            value = full_data.get("f1_score", None)
        else:
            value = full_data

        if value is not None:
            models.append(model_name)
            values.append(value)
            colors.append(MODEL_COLORS.get(model_name, 'black'))

    plt.figure()
    plt.bar(models, values, color=colors)
    plt.xlabel("Model")
    plt.ylabel("Weighted F1-score")
    plt.title(f"Full Test Set Comparison - Joint Training ({type})")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()



def plot_task_results(model_data, type='topic'):
    """
    Plot task-wise comparison of models using a grid of subplots.
    """
    task_order = TOPIC_ORDER if type == 'topic' else DATE_ORDER
    num_tasks = len(task_order)

    # Grid size (2 columns is usually readable)
    n_cols = 2
    n_rows = (num_tasks + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    for idx, task in enumerate(task_order):
        ax = axes[idx]
        models = []
        values = []
        colors = []

        for model_name, data in model_data.items():
            if 'single' in data and task in data['single']:
                models.append(model_name)
                values.append(data['single'][task])
                colors.append(MODEL_COLORS.get(model_name, 'black'))

        if models:
            ax.bar(models, values, color=colors)
            ax.set_title(task)
            ax.set_xlabel("Model")
            ax.set_ylabel("F1-score")
            ax.tick_params(axis='x', rotation=45)

    # Remove empty subplots if any
    for idx in range(num_tasks, len(axes)):
        fig.delaxes(axes[idx])

    fig.suptitle(f"Single Test Set Comparison - Joint Training ({type})", fontsize=20)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


## Results: Topic

In [ ]:
data = load_data(type='topic')
plot_full_results(data, type='topic')
plot_task_results(data, type='topic')

## Results: Date

In [ ]:
data = load_data(type='date')
plot_full_results(data, type='date')
plot_task_results(data, type='date')

In [ ]:
data = load_data(type='topic')

print("=================================")
print("FULL TEST SET (Topic): F1 RESULTS")
print("=================================\n")

for model_name, data in data.items():
    print(f"{model_name}: {data['full']}")

In [ ]:
data = load_data(type='date')

print("================================")
print("FULL TEST SET (Date): F1 RESULTS")
print("================================\n")

for model_name, data in data.items():
    print(f"{model_name}: {data['full']}")

In [ ]:
data = load_data(type='topic')

print("===================================")
print("SINGLE TEST SET (Topic): F1 RESULTS")
print("===================================\n")

for model_name, data in data.items():
    print(f"MODEL: {model_name}")
    print(data['single'])
    print("-" * (7 + len(model_name)))

In [ ]:
data = load_data(type='date')

print("==================================")
print("SINGLE TEST SET (Date): F1 RESULTS")
print("==================================\n")

for model_name, data in data.items():
    print(f"MODEL: {model_name}")
    print(data['single'])
    print("-" * (7 + len(model_name)))